# Simple Sudoku Solver

Constraint elimination: every empty cell starts with the candidates 1-9, and any digit already placed in the cell's
row, column or 3 × 3 box is crossed off. A cell left with one candidate is filled in, which may unlock others;
repeat until nothing changes.

Elimination alone solves easy puzzles. When it stalls, `solve` guesses: it picks the open cell with the fewest
candidates, tries each one, and backtracks on a contradiction.

In [1]:
puzzle = [0,0,4,0,5,0,0,0,0,
          9,0,0,7,3,4,6,0,0,
          0,0,3,0,2,1,0,4,9,
          0,3,5,0,9,0,4,8,0,
          0,9,0,0,0,0,0,3,0,
          0,7,6,0,1,0,9,2,0,
          3,1,0,9,7,0,2,0,0,
          0,0,9,1,8,2,0,0,3,
          0,0,0,0,6,0,1,0,0]

puzzle1 = [0,9,0,3,0,6,1,0,0,
           0,0,0,0,0,0,0,0,0,
           0,0,0,0,0,0,0,5,4,
           3,8,0,0,0,0,0,0,0,
           0,0,9,1,0,0,0,0,5,
           0,0,1,0,0,0,9,0,0,
           5,0,0,0,0,3,0,0,0,
           9,6,2,8,0,5,0,0,3,
           1,0,0,0,0,0,0,0,0]

In [2]:
DIGITS = set(range(1, 10))
# PEERS[i] = indexes of the 20 cells sharing a row, column or box with cell i
PEERS = [
    {j for j in range(81) if j != i and (j // 9 == i // 9 or j % 9 == i % 9
                                         or (j // 27 == i // 27 and j % 9 // 3 == i % 9 // 3))}
    for i in range(81)
]


def eliminate(grid):
    """Fill naked singles until nothing changes. Returns False on a contradiction.

    grid: list of 81 sets of candidates (a solved cell is a 1-element set); modified in place.
    """
    changed = True
    while changed:
        changed = False
        for i, cands in enumerate(grid):
            if len(cands) > 1:
                cands -= {next(iter(grid[j])) for j in PEERS[i] if len(grid[j]) == 1}
                changed |= len(cands) == 1
            if not cands:
                return False
    return all(len(grid[i]) > 1 or grid[i] != grid[j] for i in range(81) for j in PEERS[i])


def solve(puzzle, verbose=False):
    """Solve a puzzle given as 81 ints (0 = empty). Returns 81 ints, or None if unsolvable."""
    grid = [{d} if d else set(DIGITS) for d in puzzle]
    if not eliminate(grid):
        return None
    if verbose:
        print(f"after elimination: {sum(len(c) == 1 for c in grid)} of 81 cells known")
    return _search(grid)


def _search(grid):
    open_cells = [i for i in range(81) if len(grid[i]) > 1]
    if not open_cells:
        return [next(iter(c)) for c in grid]
    i = min(open_cells, key=lambda k: len(grid[k]))
    for d in sorted(grid[i]):
        trial = [set(c) for c in grid]
        trial[i] = {d}
        if eliminate(trial) and (result := _search(trial)):
            return result
    return None


def print_puzzle(puzzle):
    line = "+-------+-------+-------+"
    print(line)
    for r in range(9):
        cells = [str(d or ".") for d in puzzle[r * 9:(r + 1) * 9]]
        print("| " + " | ".join(" ".join(cells[c:c + 3]) for c in (0, 3, 6)) + " |")
        if r % 3 == 2:
            print(line)

In [3]:
for p in (puzzle, puzzle1):
    print_puzzle(p)
    solution = solve(p, verbose=True)
    print_puzzle(solution)
    assert all(a in (0, b) for a, b in zip(p, solution))                       # keeps the givens
    assert all(solution[i] != solution[j] for i in range(81) for j in PEERS[i])  # valid
    print()

+-------+-------+-------+
| . . 4 | . 5 . | . . . |
| 9 . . | 7 3 4 | 6 . . |
| . . 3 | . 2 1 | . 4 9 |
+-------+-------+-------+
| . 3 5 | . 9 . | 4 8 . |
| . 9 . | . . . | . 3 . |
| . 7 6 | . 1 . | 9 2 . |
+-------+-------+-------+
| 3 1 . | 9 7 . | 2 . . |
| . . 9 | 1 8 2 | . . 3 |
| . . . | . 6 . | 1 . . |
+-------+-------+-------+
after elimination: 81 of 81 cells known
+-------+-------+-------+
| 2 6 4 | 8 5 9 | 3 1 7 |
| 9 8 1 | 7 3 4 | 6 5 2 |
| 7 5 3 | 6 2 1 | 8 4 9 |
+-------+-------+-------+
| 1 3 5 | 2 9 7 | 4 8 6 |
| 8 9 2 | 5 4 6 | 7 3 1 |
| 4 7 6 | 3 1 8 | 9 2 5 |
+-------+-------+-------+
| 3 1 8 | 9 7 5 | 2 6 4 |
| 6 4 9 | 1 8 2 | 5 7 3 |
| 5 2 7 | 4 6 3 | 1 9 8 |
+-------+-------+-------+

+-------+-------+-------+
| . 9 . | 3 . 6 | 1 . . |
| . . . | . . . | . . . |
| . . . | . . . | . 5 4 |
+-------+-------+-------+
| 3 8 . | . . . | . . . |
| . . 9 | 1 . . | . . 5 |
| . . 1 | . . . | 9 . . |
+-------+-------+-------+
| 5 . . | . . 3 | . . . |
| 9 6 2 | 8 . 5 | . . 3